# Test resources: NeurIPS 2022 CD34+ HSPCs

Mobilised peripheral blood CD34+ hematopoietic stem and progenitor cells (HSPCs) from four healthy human donors, profiled over a 10-day time course for the [NeurIPS 2022 multimodal single-cell integration competition](https://www.kaggle.com/competitions/open-problems-multimodal) on Kaggle:

- **CITE-seq**: gene expression (GEX) together with the abundance of 140 cell surface proteins (ADT), on days 2, 3, 4 and 7.
- **Multiome**: gene expression together with chromatin accessibility (ATAC), on days 2, 3, 4, 7 and 10.

The cells are annotated with seven progenitor cell types. The cell types of the Multiome cells in the private test set of the competition were not released, and are labelled `hidden`.

The source files are stored in the private LaminDB instance [`openproblems/private`](https://lamin.ai/openproblems/private), so you need access to it to run this notebook. The test resources themselves are public.

In [ ]:
import mudata as md
import lamindb as ln

import helpers

helpers.check_instance()
ln.track("fTZVtuOVzHtH", project=helpers.PROJECT)
helpers.link_to_github("scripts/create_test_resources/neurips2022_pbmc.ipynb")

## Source files

The training and test data of the competition, merged per modality.

In [ ]:
inputs = {
    file: helpers.fetch_private(f"neurips2022/{file}.h5ad")
    for file in ["cite_rna_merged", "cite_prot_merged", "multiome_rna_merged", "multiome_atac_merged"]
}

## Select cells and features

For both datasets we subsample 600 cells, evenly across batches, and 1500 features of each modality (or all of them, for the surface proteins of CITE-seq). The load workflow stores the result as a raw MuData store, with modalities `rna` and `prot` or `atac` and the cell metadata in the global `obs`.

In [ ]:
common = {
    "dataset_url": "https://www.kaggle.com/competitions/open-problems-multimodal/data",
    "dataset_reference": "lance2024predicting",
}

raw_dir = helpers.run_workflow("multimodal/load_openproblems_neurips2022_pbmc", {
    "param_list": [
        {
            "id": "openproblems_neurips2022/pbmc_cite",
            "input_mod1": str(inputs["cite_rna_merged"]),
            "input_mod2": str(inputs["cite_prot_merged"]),
            "mod1": "GEX",
            "mod2": "ADT",
            "dataset_name": "OpenProblems NeurIPS2022 CITE-Seq",
            "dataset_summary": "Single-cell CITE-Seq (GEX+ADT) data of mobilised peripheral blood CD34+ hematopoietic stem and progenitor cells from 4 healthy human donors, over 4 days.",
            "dataset_description": "Single-cell CITE-Seq data of mobilised peripheral blood CD34+ hematopoietic stem and progenitor cells (HSPCs) from 4 healthy human donors, measured on days 2, 3, 4 and 7 of a 10-day time course. Gene expression was measured together with the abundance of 140 cell surface proteins. The dataset was generated for the Multimodal Single-Cell Integration competition at NeurIPS 2022, in which the abundance of the proteins had to be predicted from gene expression on a later day. The cells were annotated with seven progenitor cell types.",
            **common,
        },
        {
            "id": "openproblems_neurips2022/pbmc_multiome",
            "input_mod1": str(inputs["multiome_rna_merged"]),
            "input_mod2": str(inputs["multiome_atac_merged"]),
            "mod1": "GEX",
            "mod2": "ATAC",
            "dataset_name": "OpenProblems NeurIPS2022 Multiome",
            "dataset_summary": "Single-cell Multiome (GEX+ATAC) data of mobilised peripheral blood CD34+ hematopoietic stem and progenitor cells from 4 healthy human donors, over 5 days.",
            "dataset_description": "Single-cell Multiome data of mobilised peripheral blood CD34+ hematopoietic stem and progenitor cells (HSPCs) from 4 healthy human donors, measured on days 2, 3, 4, 7 and 10 of a 10-day time course with the 10X Multiome Gene Expression and Chromatin Accessibility kit. The dataset was generated for the Multimodal Single-Cell Integration competition at NeurIPS 2022, in which gene expression had to be predicted from chromatin accessibility on a later day. The cells were annotated with seven progenitor cell types; the cell types of the cells in the private test set were not released.",
            **common,
        },
    ],
    "do_subsample": True,
    "even": True,
    "n_obs": 600,
    "n_vars": 1500,
    "seed": 123,
    "output_raw": "$id/raw.mudata.zarr",
    "output_state": "$id/state.yaml",
}, highmem="100.GB")

## Process

Each modality is normalised in the way that suits it: gene expression to log counts per 10,000 (log CP10k), protein abundance (ADT) with a centred log-ratio transformation (CLR) and chromatin accessibility (ATAC) with TF-IDF. For each modality we then compute a truncated SVD embedding and select highly variable features. The processed modalities are combined into one MuData store again.

In [ ]:
output_dir = helpers.run_workflow("multimodal/process_multimodal", {
    "param_list": [
        {"id": f"openproblems_neurips2022/{dataset}", "input": str(raw_dir / "openproblems_neurips2022" / dataset / "raw.mudata.zarr")}
        for dataset in ["pbmc_cite", "pbmc_multiome"]
    ],
    "output_dataset": "$id/dataset.mudata.zarr",
    "output_state": "$id/state_process.yaml",
}, output_dir=raw_dir)

## Result

In [ ]:
for dataset in ["pbmc_cite", "pbmc_multiome"]:
    for file in ["raw", "dataset"]:
        print(dataset, file, md.read_zarr(output_dir / "openproblems_neurips2022" / dataset / f"{file}.mudata.zarr"), sep="\n")

In [ ]:
helpers.save_collection(
    output_dir / "openproblems_neurips2022",
    [f"{dataset}/{file}.mudata.zarr" for dataset in ["pbmc_cite", "pbmc_multiome"] for file in ["raw", "dataset"]],
    key="resources_test/datasets/openproblems_neurips2022",
    description="NeurIPS 2022 CD34+ HSPC CITE-seq and Multiome, subsampled to 600 cells and 1500 features per modality",
    modality="multimodal",
)

In [ ]:
ln.finish()